In [1]:
!pip install transformers torch pandas tqdm -q

In [2]:
import torch
import os
import pandas as pd
import numpy as np
from itertools import permutations
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
train_df = pd.read_csv('/content/drive/MyDrive/sentence_prediction/train.csv')
test_df  = pd.read_csv('/content/drive/MyDrive/sentence_prediction/test.csv')

print(f"train: {len(train_df)}개")
print(f"test : {len(test_df)}개")
print()
print("샘플 확인:")
print(train_df.head(2))

train: 7351개
test : 1780개

샘플 확인:
           ID                                         sentence_0  \
0  TRAIN_0000                 블록체인 기술은 투표 과정의 투명성을 크게 향상시킬 수 있다.   
1  TRAIN_0001  줄거리 자동 생성의 인공지능 알고리즘은 대량의 텍스트 데이터를 분석하여 핵심 정보를...   

                                          sentence_1  \
0  이러한 특성은 유권자들에게 신뢰를 제공하며, 민주적 참여를 촉진하는 데 기여할 수 있다.   
1     결과적으로, 이러한 기술은 사용자에게 신속하고 효율적인 정보 전달을 가능하게 한다.   

                                          sentence_2  \
0  결과적으로 블록체인 기반의 투표 시스템은 공정하고 신뢰할 수 있는 선거 환경을 조성...   
1     생성된 줄거리는 원본 텍스트의 의미를 유지하면서도 간결하게 요약된 형태로 제공된다.   

                                          sentence_3  answer_0  answer_1  \
0       각 투표는 변경 불가능한 기록으로 저장되어 조작의 가능성을 원천적으로 차단한다.         0         3   
1  이 알고리즘은 자연어 처리 기술을 활용하여 문맥을 이해하고, 주요 사건과 등장인물을...         0         3   

   answer_2  answer_3  
0         1         2  
1         2         1  


In [5]:
MODEL_NAME = "skt/kogpt2-base-v2"

print(f"로드 중: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
print("로드 완료!")

로드 중: skt/kogpt2-base-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/513M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


로드 완료!


In [6]:
SEPARATOR = " "     # 실험 1-1: 공백
# SEPARATOR = "\n"  # 실험 1-2: 줄바꿈
# SEPARATOR = ". "  # 실험 1-3: 마침표

print(f"현재 연결방식: {repr(SEPARATOR)}")

sample = train_df.iloc[0]
sentences = [sample['sentence_0'], sample['sentence_1'],
             sample['sentence_2'], sample['sentence_3']]
print("\n연결 결과 예시:")
print(SEPARATOR.join(sentences))

현재 연결방식: ' '

연결 결과 예시:
블록체인 기술은 투표 과정의 투명성을 크게 향상시킬 수 있다. 이러한 특성은 유권자들에게 신뢰를 제공하며, 민주적 참여를 촉진하는 데 기여할 수 있다. 결과적으로 블록체인 기반의 투표 시스템은 공정하고 신뢰할 수 있는 선거 환경을 조성할 잠재력을 지닌다. 각 투표는 변경 불가능한 기록으로 저장되어 조작의 가능성을 원천적으로 차단한다.


In [7]:
def calc_ppl(text):
    """
    텍스트의 Perplexity 계산
    낮을수록 = 모델이 자연스럽다고 판단
    """
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])

    ppl = torch.exp(outputs.loss).item()
    return ppl


def find_best_order(sentences, separator):
    """
    24가지 순열 중 PPL 최소 순서 반환
    """
    best_ppl   = float('inf')
    best_order = None

    for perm in permutations(range(4)):
        text = separator.join([sentences[i] for i in perm])
        ppl  = calc_ppl(text)

        if ppl < best_ppl:
            best_ppl   = ppl
            best_order = list(perm)

    return best_order

In [8]:
# 전체 돌리기 전에 5개만 먼저 테스트
print("=== 소규모 테스트 (5개) ===\n")

correct = 0
for i in range(5):
    row       = train_df.iloc[i]
    sentences = [row['sentence_0'], row['sentence_1'],
                 row['sentence_2'], row['sentence_3']]
    answer    = [int(row['answer_0']), int(row['answer_1']),
                 int(row['answer_2']), int(row['answer_3'])]

    pred = find_best_order(sentences, SEPARATOR)

    is_correct = (pred == answer)
    if is_correct:
        correct += 1

    print(f"샘플 {i+1}")
    print(f"  예측: {pred}")
    print(f"  정답: {answer}")
    print(f"  결과: {'✅' if is_correct else '❌'}")

print(f"\n5개 중 {correct}개 정답")

=== 소규모 테스트 (5개) ===



model.safetensors:   0%|          | 0.00/513M [00:00<?, ?B/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


샘플 1
  예측: [1, 3, 0, 2]
  정답: [0, 3, 1, 2]
  결과: ❌
샘플 2
  예측: [1, 3, 2, 0]
  정답: [0, 3, 2, 1]
  결과: ❌
샘플 3
  예측: [1, 2, 3, 0]
  정답: [3, 2, 1, 0]
  결과: ❌
샘플 4
  예측: [0, 1, 3, 2]
  정답: [2, 0, 1, 3]
  결과: ❌
샘플 5
  예측: [2, 0, 3, 1]
  정답: [1, 3, 0, 2]
  결과: ❌

5개 중 0개 정답


In [9]:
EXP_NAME = f"{MODEL_NAME.split('/')[-1]}_{repr(SEPARATOR)}"
print(f"실험명: {EXP_NAME}")
print(f"전체 {len(test_df)}개 예측 시작...\n")

predictions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    sentences = [row['sentence_0'], row['sentence_1'],
                 row['sentence_2'], row['sentence_3']]
    pred = find_best_order(sentences, SEPARATOR)
    predictions.append(pred)

print("\n예측 완료!")

실험명: kogpt2-base-v2_' '
전체 1780개 예측 시작...



100%|██████████| 1780/1780 [08:55<00:00,  3.32it/s]


예측 완료!


In [10]:
# test_df엔 정답이 없으므로 train_df로 검증
print("=== train 데이터로 정확도 검증 ===\n")

train_predictions = []

for _, row in tqdm(train_df.iterrows(), total=len(train_df)):
    sentences = [row['sentence_0'], row['sentence_1'],
                 row['sentence_2'], row['sentence_3']]
    pred = find_best_order(sentences, SEPARATOR)
    train_predictions.append(pred)

# 정확도 계산
correct = 0
for i, pred in enumerate(train_predictions):
    answer = [int(train_df.iloc[i]['answer_0']), int(train_df.iloc[i]['answer_1']),
              int(train_df.iloc[i]['answer_2']), int(train_df.iloc[i]['answer_3'])]
    if pred == answer:
        correct += 1

acc = correct / len(train_df)
print(f"\n모델     : {MODEL_NAME}")
print(f"연결방식 : {repr(SEPARATOR)}")
print(f"정확도   : {acc:.4f} ({correct}/{len(train_df)})")

result = {
    'model'    : [MODEL_NAME],
    'separator': [repr(SEPARATOR)],
    'accuracy' : [acc]
}

result_df   = pd.DataFrame(result)
result_path = '/content/drive/MyDrive/sentence_prediction/results/experiments.csv'

# 파일 있으면 이어쓰기, 없으면 새로 생성
if os.path.exists(result_path):
    existing = pd.read_csv(result_path)
    result_df = pd.concat([existing, result_df], ignore_index=True)

result_df.to_csv(result_path, index=False)
print("결과 저장 완료!")
print(result_df)

=== train 데이터로 정확도 검증 ===



100%|██████████| 7351/7351 [36:18<00:00,  3.37it/s]



모델     : skt/kogpt2-base-v2
연결방식 : ' '
정확도   : 0.0316 (232/7351)
결과 저장 완료!
                model separator  accuracy
0  skt/kogpt2-base-v2       ' '   0.03156
